> ⚠️ **Superseded.** These confidence intervals use Newey-West/HAC errors and are no longer the team's numbers. The corrected HC3 results are in [`regression_did_hc3_and_anticipation.ipynb`](regression_did_hc3_and_anticipation.ipynb). Kept for the record; don't cite these CIs.

# Issue #29 — Simple regression DiD ATT and 95% CI

Uses only **United States + Canada** and reuses `src/simple_did.py` helpers. The treatment-effect coefficient is the interaction `pilot_market:post_launch`. Standard errors are Newey-West/HAC via `sd.smf_ols()`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Works from notebooks/simple_did or notebooks/. Finds repo root by looking for src/simple_did.py.
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "src" / "simple_did.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find repo root containing src/simple_did.py")

sys.path.append(str(PROJECT_ROOT / "src"))
import simple_did as sd

TREATED = "United States"
CONTROL = "Canada"
METRIC = sd.METRIC  # revenue_per_session

df = sd.load_panel()
us_ca = df[df["market"].isin([TREATED, CONTROL])].copy()

print("Project root:", PROJECT_ROOT)
print("Metric:", METRIC)
print("US + Canada rows:", len(us_ca))

Project root: .
Metric: revenue_per_session
US + Canada rows: 208


In [2]:
formula = f"{METRIC} ~ pilot_market + post_launch + pilot_market:post_launch"
simple_model = sd.smf_ols(formula, us_ca)  # Newey-West/HAC errors, maxlags=8 by default
term = "pilot_market:post_launch"
ci = simple_model.conf_int().loc[term]

simple_att = float(simple_model.params[term])
simple_se = float(simple_model.bse[term])
simple_p = float(simple_model.pvalues[term])
simple_ci_low = float(ci[0])
simple_ci_high = float(ci[1])

simple_result = pd.DataFrame([{
    "model": "simple_regression_did",
    "term": term,
    "att": simple_att,
    "std_err": simple_se,
    "p_value": simple_p,
    "ci_low": simple_ci_low,
    "ci_high": simple_ci_high,
    "nobs": int(simple_model.nobs),
}])

simple_result

                   model                      term  ...   ci_high  nobs
0  simple_regression_did  pilot_market:post_launch  ...  0.512167   208

[1 rows x 8 columns]

In [3]:
sd.save_estimate(
    method="simple_regression_did",
    estimate=simple_att,
    issue=29,
    treated=TREATED,
    control=CONTROL,
    metric=METRIC,
    window="all_pre_vs_post",
    ci_low=simple_ci_low,
    ci_high=simple_ci_high,
    notes="Simple regression DiD using US + Canada only. Coefficient on pilot_market:post_launch; OLS with Newey-West/HAC errors via sd.smf_ols(maxlags=8).",
)

print(f"Simple regression DiD ATT: {simple_att:+.4f} RPS")
print(f"95% CI: [{simple_ci_low:+.4f}, {simple_ci_high:+.4f}]")
print(f"p-value: {simple_p:.4f}")

Simple regression DiD ATT: +0.2459 RPS
95% CI: [-0.0203, +0.5122]
p-value: 0.0703


## Result

Simple regression DiD ATT = **+0.2459 RPS**. 95% CI = **[-0.0203, +0.5122]**. p = **0.0703**.

This satisfies issue #29: fitted on US + Canada only, reported ATT + CI, and saves one estimate row through `sd.save_estimate()`.